In [1]:
import pandas as pd

rows = [
    ("C01","Ravi Kumar","Pune","MH"),
    ("C02","Anita Sharma","Mumbai","MH"),
     ("C03","Rahul Verma","Delhi","DL")
]

df = pd.DataFrame(rows, columns=["customer_id","customer_name","city","state"])
csv_path = "D:/DataMining_coursework/Customers.csv"
df.to_csv(csv_path, index=False)
print(f"Saved source data to: {csv_path}")
df.head()

Saved source data to: D:/DataMining_coursework/Customers.csv


,customer_id,customer_name,city,state
0,C01,Ravi Kumar,Pune,MH
1,C02,Anita Sharma,Mumbai,MH
2,C03,Rahul Verma,Delhi,DL


In [2]:
import pandas as pd

rows = [
    ("P01","Laptop","Electronics","55000"),
    ("P02","Mobile","Electronics","20000"),
     ("P03","Chair","Furniture","3000")
]

df = pd.DataFrame(rows, columns=["product_id","product_name","category","price"])
csv_path = "D:/DataMining_coursework/Products.csv"
df.to_csv(csv_path, index=False)
print(f"Saved source data to: {csv_path}")
df.head()

Saved source data to: D:/DataMining_coursework/Products.csv


,product_id,product_name,category,price
0,P01,Laptop,Electronics,55000
1,P02,Mobile,Electronics,20000
2,P03,Chair,Furniture,3000


In [3]:

import pandas as pd

rows = [
("S01","C01","P01","2024-01-10","1"),
("S02","C02","P02","2024-01-12","2"),
("S03","C03","P03","2024-02-05,4")
]

df = pd.DataFrame(rows, columns=["sale_id","customer_id","product_id","sale_date","quantity"])
csv_path = "D:/DataMining_coursework/Sales.csv"
df.to_csv(csv_path, index=False)
print(f"Saved source data to: {csv_path}")
df.head()

Saved source data to: D:/DataMining_coursework/Sales.csv


,sale_id,customer_id,product_id,sale_date,quantity
0,S01,C01,P01,2024-01-10,1
1,S02,C02,P02,2024-01-12,2
2,S03,C03,P03,"2024-02-05,4",None


In [4]:
import pandas as pd
from pathlib import Path

def resolve_csv(file_name: str) -> Path:
    candidates = [
        Path(file_name),
        Path("experiment1") / file_name,
        Path("D:/DataMining_coursework") / file_name,
        Path("D:/DataMining_coursework/experiment1") / file_name,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {file_name} in expected locations.")

customers = pd.read_csv(resolve_csv("Customers.csv"))
products = pd.read_csv(resolve_csv("Products.csv"))
sales = pd.read_csv(resolve_csv("Sales.csv"))

# Clean potentially inconsistent quantity values before numeric conversion.
sales["quantity"] = (
    sales["quantity"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
    .fillna(0)
    .astype(int)
)

sales["sale_date"] = pd.to_datetime(sales["sale_date"], errors="coerce")

fact = (
    sales.merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
)
fact["revenue"] = fact["quantity"] * pd.to_numeric(fact["price"], errors="coerce").fillna(0)

print("Joined fact table:")
display(fact)

# 1) ROLL-UP: Aggregate from city level to state level (higher hierarchy).
roll_up = (
    fact.groupby(["state", "city"], as_index=False)["revenue"]
    .sum()
    .sort_values(["state", "city"])
    .rename(columns={"revenue": "city_revenue"})
)
state_totals = (
    fact.groupby("state", as_index=False)["revenue"]
    .sum()
    .sort_values("state")
    .rename(columns={"revenue": "state_revenue"})
)
print("\nROLL-UP (City -> State):")
display(roll_up)
display(state_totals)

# 2) ROLL-DOWN (Drill-Down): From state to city to customer.
drill_down = (
    fact.groupby(["state", "city", "customer_name"], as_index=False)["revenue"]
    .sum()
    .sort_values(["state", "city", "customer_name"])
    .rename(columns={"revenue": "customer_revenue"})
)
print("\nROLL-DOWN (State -> City -> Customer):")
display(drill_down)

# 3) SLICE: Fix one dimension value (category = Electronics).
slice_df = fact[fact["category"] == "Electronics"][[
    "sale_id", "customer_name", "product_name", "category", "quantity", "revenue"
]]
print("\nSLICE (Category = Electronics):")
display(slice_df)

# 4) DICE: Filter on multiple dimensions (state and category).
dice_df = fact[
    (fact["state"].isin(["MH", "DL"]))
    & (fact["category"].isin(["Electronics", "Furniture"]))
    & (fact["quantity"] >= 1)
][[
    "sale_id", "state", "city", "category", "quantity", "revenue"
]]
print("\nDICE (State in {MH, DL} and Category in {Electronics, Furniture}):")
display(dice_df)

# 5) PIVOT: Category-wise revenue by state.
pivot_table = pd.pivot_table(
    fact,
    index="state",
    columns="category",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Grand Total",
)
print("\nPIVOT TABLE (Revenue by State x Category):")
display(pivot_table)

Joined fact table:


,sale_id,customer_id,product_id,sale_date,quantity,customer_name,city,state,product_name,category,price,revenue
0,S01,C01,P01,2024-01-10,1,Ravi Kumar,Pune,MH,Laptop,Electronics,55000,55000
1,S02,C02,P02,2024-01-12,2,Anita Sharma,Mumbai,MH,Mobile,Electronics,20000,40000
2,S03,C03,P03,NaT,0,Rahul Verma,Delhi,DL,Chair,Furniture,3000,0



ROLL-UP (City -> State):


,state,city,city_revenue
0,DL,Delhi,0
1,MH,Mumbai,40000
2,MH,Pune,55000


,state,state_revenue
0,DL,0
1,MH,95000


...

,sale_id,customer_name,product_name,category,quantity,revenue
0,S01,Ravi Kumar,Laptop,Electronics,1,55000
1,S02,Anita Sharma,Mobile,Electronics,2,40000



DICE (State in {MH, DL} and Category in {Electronics, Furniture}):


,sale_id,state,city,category,quantity,revenue
0,S01,MH,Pune,Electronics,1,55000
1,S02,MH,Mumbai,Electronics,2,40000



PIVOT TABLE (Revenue by State x Category):


category,Electronics,Furniture,Grand Total
state,,,
DL,0,0,0
MH,95000,0,95000
Grand Total,95000,0,95000
